In [ ]:
%pip install -r requirements.txt

In [ ]:
RAW_DIR       = './data/raw'
PROCESSED_DIR = './data/processed'

In [ ]:
import tarfile
from pathlib import Path

import requests
from tqdm.notebook import tqdm

URLS = {
    "summary.csv": (
        "https://libdrive.ethz.ch/index.php/s/X5vOBNSITAG5vzM/download"
        "?path=/&files=summary.csv"
    ),
    "structures.tar.gz": (
        "https://libdrive.ethz.ch/index.php/s/X5vOBNSITAG5vzM/download"
        "?path=/&files=structures.tar.gz"
    ),
}


def download_file(url: str, dest: Path) -> None:
    if dest.exists():
        print(f"[skip] {dest.name} already exists")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".tmp")
    try:
        with requests.get(url, stream=True, timeout=30) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(tmp, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, desc=dest.name, leave=True
            ) as bar:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    bar.update(len(chunk))
        tmp.rename(dest)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


data_dir = Path(RAW_DIR)
data_dir.mkdir(parents=True, exist_ok=True)

download_file(URLS["summary.csv"], data_dir / "summary.csv")

structures_dir = data_dir / "structures"
tar_path = data_dir / "structures.tar.gz"

if structures_dir.exists():
    print("[skip] structures/ already extracted")
else:
    download_file(URLS["structures.tar.gz"], tar_path)
    with tarfile.open(tar_path, "r:gz") as tar:
        with tqdm(unit=" files", desc="Extracting") as bar:
            for member in tar:
                tar.extract(member, data_dir, filter="data")
                bar.update(1)
    print("Extraction complete.")
    tar_path.unlink()
    print(f"Deleted {tar_path.name}")

In [ ]:
import json
import math
import multiprocessing
import os
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolTransforms
from torch_geometric.data import Data
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
RDLogger.DisableLog("rdApp.warning")

PROP_COLS = [
    "DFT_HOMO_ENERGY",
    "DFT_LUMO_ENERGY",
    "DFT_HOMO_LUMO_GAP",
    "DFT_TOTAL_ENERGY",
    "DFT_DIPOLE_TOT",
]
ELEMENTS = ["C", "H", "O", "N", "S", "P", "F", "Cl", "Br", "I"]

BOND_TYPE_MAP = {
    1.0: [1, 0, 0, 0],
    2.0: [0, 1, 0, 0],
    3.0: [0, 0, 1, 0],
    1.5: [0, 0, 0, 1],
}

from rdkit.Chem import rdchem
STEREO_MAP = {
    rdchem.BondStereo.STEREONONE:  [1, 0, 0, 0, 0],
    rdchem.BondStereo.STEREOCIS:   [0, 1, 0, 0, 0],
    rdchem.BondStereo.STEREOTRANS: [0, 0, 1, 0, 0],
    rdchem.BondStereo.STEREOE:     [0, 0, 0, 1, 0],
    rdchem.BondStereo.STEREOZ:     [0, 0, 0, 0, 1],
}
STEREO_DEFAULT = [1, 0, 0, 0, 0]


def _pick_neighbor(mol, center_idx, exclude_idx):
    neighbors = [
        a for a in mol.GetAtomWithIdx(center_idx).GetNeighbors()
        if a.GetIdx() != exclude_idx
    ]
    if not neighbors:
        return None
    return max(neighbors, key=lambda a: (a.GetAtomicNum(), -a.GetIdx())).GetIdx()


def _compute_composition(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    counts = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    a = [counts.get(e, 0) for e in ELEMENTS]
    a.append(mol.GetNumAtoms())
    return a


def _process_sdf(args):
    sdf_path_str, row = args
    chembl_id = Path(sdf_path_str).parent.name
    try:
        supplier = Chem.SDMolSupplier(sdf_path_str, removeHs=False, sanitize=True)
        mol = supplier[0] if supplier and len(supplier) > 0 else None
    except Exception:
        return None
    if mol is None:
        return None
    n_atoms = mol.GetNumAtoms()
    if n_atoms < 3 or n_atoms > 100:
        return None
    if mol.GetNumConformers() == 0:
        return None
    conf = mol.GetConformer()
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    z_list, charge_list, chirality_list = [], [], []
    for atom in mol.GetAtoms():
        z_list.append(atom.GetAtomicNum())
        charge_list.append(float(atom.GetFormalCharge()))
        if atom.HasProp("_CIPCode"):
            cip = atom.GetProp("_CIPCode")
            chirality_list.append([1, 0, 0] if cip == "R" else [0, 1, 0])
        else:
            chirality_list.append([0, 0, 1])
    src_list, dst_list = [], []
    bt_list, bs_list, dih_list = [], [], []
    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        bt_val = bond.GetBondTypeAsDouble()
        bt = BOND_TYPE_MAP.get(bt_val, [1, 0, 0, 0])
        bs = STEREO_MAP.get(bond.GetStereo(), STEREO_DEFAULT)
        dih = 0.0
        if bt_val == 1.0 and not bond.IsInRing():
            i = _pick_neighbor(mol, u, v)
            j = _pick_neighbor(mol, v, u)
            if i is not None and j is not None:
                try:
                    deg = rdMolTransforms.GetDihedralDeg(conf, i, u, v, j)
                    dih = float(deg) * math.pi / 180.0
                except Exception:
                    dih = 0.0
        for s, d in ((u, v), (v, u)):
            src_list.append(s)
            dst_list.append(d)
            bt_list.append(bt)
            bs_list.append(bs)
            dih_list.append(dih)
    return {
        "z":           z_list,
        "pos":         conf.GetPositions().tolist(),
        "charge":      charge_list,
        "chirality":   chirality_list,
        "src":         src_list,
        "dst":         dst_list,
        "bond_type":   bt_list,
        "bond_stereo": bs_list,
        "dihedral":    dih_list,
        "p_raw":       row["p_raw"],
        "a":           row["a"],
        "chembl_id":   chembl_id,
        "smiles":      row["smiles"],
    }


def _dict_to_data(d):
    return Data(
        z=torch.tensor(d["z"], dtype=torch.long),
        pos=torch.tensor(d["pos"], dtype=torch.float),
        charge=torch.tensor(d["charge"], dtype=torch.float),
        chirality=torch.tensor(d["chirality"], dtype=torch.float),
        edge_index=torch.tensor([d["src"], d["dst"]], dtype=torch.long),
        bond_type=torch.tensor(d["bond_type"], dtype=torch.float),
        bond_stereo=torch.tensor(d["bond_stereo"], dtype=torch.float),
        dihedral=torch.tensor(d["dihedral"], dtype=torch.float),
        p_raw=torch.tensor(d["p_raw"], dtype=torch.float),
        a=torch.tensor(d["a"], dtype=torch.float),
        chembl_id=d["chembl_id"],
        smiles=d["smiles"],
    )

In [ ]:
import subprocess

raw_dir = Path(RAW_DIR)
out_dir = Path(PROCESSED_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print("Loading summary.csv ...")
df = pd.read_csv(raw_dir / "summary.csv", usecols=["chembl_id", "smiles"] + PROP_COLS)
df = df.dropna()
print(f"  {len(df):,} rows after NaN removal")

stats = {
    col: {"mean": float(df[col].mean()), "std": float(df[col].std())}
    for col in PROP_COLS
}
with open(out_dir / "stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print(f"  Saved stats -> {out_dir / 'stats.json'}")

print("Computing composition vectors ...")
mol_data = {}
skipped_smiles = 0
for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
    a = _compute_composition(row["smiles"])
    if a is None:
        skipped_smiles += 1
        continue
    mol_data[row["chembl_id"]] = {
        "smiles": row["smiles"],
        "p_raw": [float(row[c]) for c in PROP_COLS],
        "a": a,
    }
print(f"  {len(mol_data):,} molecules ({skipped_smiles} invalid SMILES skipped)")
del df

structures_dir = raw_dir / "structures"
print(f"Scanning {structures_dir} for SDF files ...")
result = subprocess.run(
    ["find", str(structures_dir), "-name", "*.sdf", "-type", "f"],
    capture_output=True, text=True
)
sdf_paths = [Path(p) for p in result.stdout.strip().split("\n") if p]
del result
print(f"  Found {len(sdf_paths):,} SDF files")

tasks, skipped_no_match = [], 0
for p in sdf_paths:
    cid = p.parent.name
    if cid in mol_data:
        tasks.append((str(p), mol_data[cid]))
    else:
        skipped_no_match += 1
del sdf_paths, mol_data
print(f"  {len(tasks):,} tasks ({skipped_no_match:,} skipped: no CSV match)")

n_workers = os.cpu_count() or 4
chunksize  = 256
CHUNK_SIZE = 50_000
print(f"Processing with {n_workers} workers (chunksize={chunksize}) ...")

data_list, filtered, chunk_idx, chunk_sizes = [], 0, 0, []
with multiprocessing.Pool(n_workers) as pool:
    for result in tqdm(
        pool.imap_unordered(_process_sdf, tasks, chunksize=chunksize),
        total=len(tasks),
    ):
        if result is not None:
            data_list.append(_dict_to_data(result))
        else:
            filtered += 1

        if len(data_list) >= CHUNK_SIZE:
            chunk_path = out_dir / f"data_chunk_{chunk_idx:04d}.pt"
            torch.save(data_list, chunk_path)
            print(f"  Saved {chunk_path.name} ({len(data_list):,} items)")
            chunk_sizes.append(len(data_list))
            data_list.clear()
            chunk_idx += 1

if data_list:
    chunk_path = out_dir / f"data_chunk_{chunk_idx:04d}.pt"
    torch.save(data_list, chunk_path)
    print(f"  Saved {chunk_path.name} ({len(data_list):,} items)")
    chunk_sizes.append(len(data_list))
    chunk_idx += 1

del tasks, data_list

meta = {
    "chunk_files": [f"data_chunk_{i:04d}.pt" for i in range(len(chunk_sizes))],
    "chunk_sizes": chunk_sizes,
}
with open(out_dir / "meta.json", "w") as f:
    json.dump(meta, f)
print(f"  Saved meta.json ({len(chunk_sizes)} chunks, {sum(chunk_sizes):,} total)")

total_skipped = filtered + skipped_no_match + skipped_smiles
print(f"\n필터링 제거:      {total_skipped:>10,}")
print(f"청크 파일 수:     {chunk_idx:>10,}")
print("Done.")

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset


class QMugsDataset(Dataset):
    def __init__(
        self,
        processed_path,
        split: str = "train",
        split_ratio: tuple = (0.8, 0.1, 0.1),
        seed: int = 42,
    ):
        processed_path = Path(processed_path)

        chunk_files = sorted(processed_path.glob("data_chunk_*.pt"))
        print(f"Loading {len(chunk_files)} chunks ...")
        self.data_list = []
        for f in chunk_files:
            self.data_list.extend(torch.load(f, weights_only=False))

        with open(processed_path / "stats.json") as f:
            stats = json.load(f)

        prop_cols = list(stats.keys())
        self.mean = torch.tensor([stats[c]["mean"] for c in prop_cols], dtype=torch.float)
        self.std  = torch.tensor([stats[c]["std"]  for c in prop_cols], dtype=torch.float)

        n = len(self.data_list)
        rng = np.random.default_rng(seed)
        perm = rng.permutation(n)

        train_end = int(n * split_ratio[0])
        val_end   = train_end + int(n * split_ratio[1])

        splits = {
            "train": perm[:train_end],
            "val":   perm[train_end:val_end],
            "test":  perm[val_end:],
        }
        if split not in splits:
            raise ValueError(f"split must be 'train', 'val', or 'test'; got '{split}'")
        self.indices = splits[split]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        data = self.data_list[self.indices[idx]].clone()
        data.p = (data.p_raw - self.mean) / self.std
        return data

In [ ]:
import torch
from pathlib import Path

chunk_files = sorted(Path(PROCESSED_DIR).glob("data_chunk_*.pt"))
total_chunks = len(chunk_files)
print(f"청크 파일 수: {total_chunks}")

# 첫 번째 청크만 로드해서 shape 확인
sample_chunk = torch.load(chunk_files[0], weights_only=False)
d = sample_chunk[0]
print(d)
print('z     :', d.z.shape)
print('pos   :', d.pos.shape, '| range', round(d.pos.min().item(), 2), '~', round(d.pos.max().item(), 2))
print('charge:', d.charge.shape)
print('chiral:', d.chirality.shape)
print('edges :', d.edge_index.shape)
print('p_raw :', d.p_raw)
print('a     :', d.a)

sample = sample_chunk[:1000]
print('has chirality:', any((x.chirality[:, 2] < 1).any().item() for x in sample))
print('has stereo   :', any((x.bond_stereo[:, 0] < 1).any().item() for x in sample))
del sample_chunk, sample

# split 비율 (마지막 청크 크기 반영)
last_chunk = torch.load(chunk_files[-1], weights_only=False)
total = (total_chunks - 1) * 50_000 + len(last_chunk)
del last_chunk
print(f"\n총 conformer: {total:,}")
print(f"train: {int(total*0.8):>8,} (80.0%)")
print(f"val  : {int(total*0.1):>8,} (10.0%)")
print(f"test : {total - int(total*0.8) - int(total*0.1):>8,} (10.0%)")

In [ ]:
import shutil
shutil.make_archive('processed', 'zip', PROCESSED_DIR)
print('processed.zip 생성 완료')